# Week 7 – Dimensionality Reduction
## Moneyball: IPL Player Valuation | Team 05
**Course:** 24ADI204 – Data Science and Visualization

### Objective
This notebook performs feature selection, correlation analysis, multicollinearity reduction, standard scaling and Principal Component Analysis (PCA) using the **Week 3 cleaned IPL dataset**.

**Upload only:** `IPL_Moneyball_Cleaned_Week03.csv`  
The Week 6 skewness-corrected, scaled and encoded CSV files are **not required**. All required preprocessing for this Week 7 analysis is performed in this notebook.

### Workflow
1. Load and inspect the cleaned dataset.
2. Select numerical features and handle missing values.
3. Remove constant features and inspect correlations.
4. Visualize the correlation heatmap and remove highly correlated features.
5. Standardize the selected features.
6. Apply PCA and visualize explained variance.
7. Create 2D and 3D PCA scatter plots when enough components are available.
8. Compare dimensions and save the output files.

## Step 1 – Import Libraries
Import the Python libraries required for data handling, visualization and PCA.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

sns.set_theme()

## Step 2 – Upload and Load the Cleaned Dataset
Upload only the Week 3 cleaned CSV. The code automatically reads the uploaded filename, so small filename differences will not cause an error.

In [ ]:
from google.colab import files

uploaded = files.upload()

if not uploaded:
    raise ValueError("Please upload the Week 3 cleaned IPL CSV file.")

csv_filename = next(iter(uploaded))
df = pd.read_csv(csv_filename)

print("Dataset loaded successfully:", csv_filename)
print("Dataset shape:", df.shape)
display(df.head())

## Step 3 – Understand the Dataset
Inspect data types, dimensions and missing values before feature preparation.

In [ ]:
print("Dataset Information:")
df.info()

print("\\nDataset shape:", df.shape)
print("\\nFirst five rows:")
display(df.head())

print("\\nMissing values by column:")
display(df.isnull().sum().to_frame("Missing Values"))

## Step 4 – Select Numerical Features
PCA requires numerical input. This step selects numerical columns and excludes `season`, which is a calendar identifier rather than a player-performance measurement for this analysis.

In [ ]:
numeric_df = df.select_dtypes(include=np.number).copy()
numeric_df = numeric_df.drop(columns=["season"], errors="ignore")

if numeric_df.shape[1] == 0:
    raise ValueError("No numerical feature columns were found for PCA.")

print("Numerical features selected:")
print(numeric_df.columns.tolist())
print("\\nNumber of numerical features:", numeric_df.shape[1])

## Step 5 – Handle Missing Values
Median imputation is applied to numerical features so that missing values do not prevent correlation analysis, scaling or PCA. The original `df` is not modified.

In [ ]:
imputer = SimpleImputer(strategy="median")
imputed_values = imputer.fit_transform(numeric_df)

numeric_df = pd.DataFrame(
    imputed_values,
    columns=numeric_df.columns,
    index=df.index
)

print("Missing values after imputation:", int(numeric_df.isnull().sum().sum()))
display(numeric_df.head())

## Step 6 – Feature Selection: Remove Constant Features
Features with only one unique value cannot help distinguish observations, so they are removed.

In [ ]:
selected_features = numeric_df.loc[:, numeric_df.nunique(dropna=False) > 1].copy()

print("Features before constant-feature removal:", numeric_df.shape[1])
print("Features after constant-feature removal:", selected_features.shape[1])
print("\\nSelected features:")
print(selected_features.columns.tolist())

if selected_features.shape[1] < 2:
    raise ValueError("At least two varying numerical features are required for correlation analysis and PCA.")

## Step 7 – Correlation Matrix
Calculate Pearson correlations between the selected numerical features.

In [ ]:
corr_matrix = selected_features.corr()

print("Correlation matrix:")
display(corr_matrix)

## Step 8 – Correlation Heatmap
The heatmap shows the direction and strength of linear relationships. Values near +1 or -1 indicate strong correlations; values near 0 indicate weak linear relationships.

In [ ]:
plt.figure(figsize=(14, 10))
sns.heatmap(
    corr_matrix,
    annot=True,
    cmap="coolwarm",
    fmt=".2f",
    linewidths=0.5,
    center=0
)
plt.title("Correlation Heatmap of IPL Numerical Features")
plt.tight_layout()
plt.show()

## Step 9 – Remove Multicollinearity
Use an absolute correlation threshold of **0.90**. When two features are highly correlated, one is removed to reduce redundant information. The correlation matrix is retained as evidence of the original relationships.

In [ ]:
correlation_threshold = 0.90

upper_triangle = corr_matrix.abs().where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

to_drop = [
    column
    for column in upper_triangle.columns
    if (upper_triangle[column] > correlation_threshold).any()
]

reduced_df = selected_features.drop(columns=to_drop).copy()

print("Correlation threshold:", correlation_threshold)
print("Features removed due to high correlation:", to_drop if to_drop else "None")
print("Features before multicollinearity removal:", selected_features.shape[1])
print("Features after multicollinearity removal:", reduced_df.shape[1])

if reduced_df.shape[1] < 1:
    raise ValueError("No features remain after correlation filtering. Review the correlation threshold.")

## Step 10 – Standard Scaling
PCA is sensitive to feature scale. StandardScaler transforms each feature to approximately zero mean and unit variance.

In [ ]:
scaler = StandardScaler()
scaled_data = scaler.fit_transform(reduced_df)

scaled_df = pd.DataFrame(
    scaled_data,
    columns=reduced_df.columns,
    index=reduced_df.index
)

print("Scaled dataset preview:")
display(scaled_df.head())

print("\\nScaled feature means (approximately 0):")
display(scaled_df.mean().round(3))

## Step 11 – Apply PCA
Fit PCA to all available scaled features first. This allows us to inspect the explained variance of every principal component.

In [ ]:
pca = PCA()
pca_data = pca.fit_transform(scaled_data)

pca_columns = [f"PC{i+1}" for i in range(pca_data.shape[1])]
pca_df = pd.DataFrame(pca_data, columns=pca_columns, index=df.index)

print("PCA dataset shape:", pca_df.shape)
display(pca_df.head())

## Step 12 – Explained Variance Ratio
The explained variance ratio indicates the proportion of the dataset's variance represented by each principal component.

In [ ]:
explained_variance = pca.explained_variance_ratio_

variance_table = pd.DataFrame({
    "Principal Component": pca_columns,
    "Explained Variance Ratio": explained_variance,
    "Explained Variance (%)": explained_variance * 100
})

display(variance_table.round(4))

## Step 13 – Explained Variance Plot
This bar chart compares the variance explained by each principal component.

In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(range(1, len(explained_variance) + 1), explained_variance)
plt.xlabel("Principal Components")
plt.ylabel("Explained Variance Ratio")
plt.title("Explained Variance Ratio of PCA")
plt.xticks(range(1, len(explained_variance) + 1))
plt.tight_layout()
plt.show()

## Step 14 – Cumulative Explained Variance
The cumulative plot shows the total variance retained as more components are included. The dashed line marks the 95% target.

In [ ]:
cumulative_variance = np.cumsum(explained_variance)

plt.figure(figsize=(10, 5))
plt.plot(
    range(1, len(cumulative_variance) + 1),
    cumulative_variance,
    marker="o"
)
plt.axhline(y=0.95, color="red", linestyle="--", label="95% variance target")
plt.xlabel("Number of Principal Components")
plt.ylabel("Cumulative Explained Variance")
plt.title("Cumulative Explained Variance")
plt.xticks(range(1, len(cumulative_variance) + 1))
plt.ylim(0, 1.05)
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

## Step 15 – Select the Number of Components
Select the smallest number of components that retains at least 95% of the variance. If 95% is not reached, retain all available components and clearly report the achieved variance.

In [ ]:
target_variance = 0.95
components_meeting_target = np.where(cumulative_variance >= target_variance)[0]

if len(components_meeting_target) > 0:
    n_components = int(components_meeting_target[0] + 1)
    print("95% variance target reached.")
else:
    n_components = len(cumulative_variance)
    print("95% variance target was not reached; using all available components.")

print("Number of components selected:", n_components)
print("Variance retained:", round(float(cumulative_variance[n_components - 1]), 4))
print("Variance retained (%):", round(float(cumulative_variance[n_components - 1] * 100), 2))

## Step 16 – Create the Reduced PCA Dataset
Fit PCA using the selected number of components and store the reduced representation.

In [ ]:
pca_reduced = PCA(n_components=n_components)
reduced_data = pca_reduced.fit_transform(scaled_data)

final_pca_df = pd.DataFrame(
    reduced_data,
    columns=[f"PC{i+1}" for i in range(n_components)],
    index=df.index
)

print("Reduced PCA dataset shape:", final_pca_df.shape)
display(final_pca_df.head())

## Step 17 – 2D PCA Scatter Plot
Plot the first two principal components when at least two components are available.

In [ ]:
if pca_df.shape[1] >= 2:
    plt.figure(figsize=(10, 6))
    plt.scatter(pca_df["PC1"], pca_df["PC2"], alpha=0.5)
    plt.xlabel("Principal Component 1")
    plt.ylabel("Principal Component 2")
    plt.title("2D PCA Scatter Plot")
    plt.grid(True)
    plt.tight_layout()
    plt.show()
else:
    print("2D PCA scatter plot skipped: fewer than two principal components are available.")

## Step 18 – 3D PCA Scatter Plot
A 3D plot requires at least three varying input features. The check prevents PCA from failing when fewer than three are available.

In [ ]:
if scaled_data.shape[1] >= 3:
    pca_3d = PCA(n_components=3)
    pca_3d_data = pca_3d.fit_transform(scaled_data)

    fig = plt.figure(figsize=(10, 7))
    ax = fig.add_subplot(111, projection="3d")
    ax.scatter(
        pca_3d_data[:, 0],
        pca_3d_data[:, 1],
        pca_3d_data[:, 2],
        alpha=0.5
    )
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    ax.set_zlabel("PC3")
    ax.set_title("3D PCA Scatter Plot")
    plt.tight_layout()
    plt.show()
else:
    print("3D PCA scatter plot skipped: at least three numerical features are required.")

## Step 19 – Compare Original and Reduced Dimensions
Review how the number of numerical features changes after feature selection, correlation filtering and PCA.

In [ ]:
print("Original dataset shape:", df.shape)
print("Numerical dataset shape:", numeric_df.shape)
print("After constant-feature selection:", selected_features.shape)
print("After multicollinearity removal:", reduced_df.shape)
print("After PCA reduction:", final_pca_df.shape)

print("\\nSummary:")
print(f"Numerical features before correlation filtering: {selected_features.shape[1]}")
print(f"Features after correlation filtering: {reduced_df.shape[1]}")
print(f"Principal components retained: {final_pca_df.shape[1]}")
print(f"Total variance retained: {pca_reduced.explained_variance_ratio_.sum() * 100:.2f}%")

## Step 20 – Save Output Files
Save the PCA-reduced dataset and the original correlation matrix. These outputs are generated from the cleaned CSV uploaded at the beginning of this notebook.

In [ ]:
pca_output = "IPL_PCA_Reduced_Week07.csv"
correlation_output = "IPL_Correlation_Matrix_Week07.csv"

final_pca_df.to_csv(pca_output, index=False)
corr_matrix.to_csv(correlation_output)

print("Week 7 output files saved successfully:")
print(pca_output)
print(correlation_output)

## Step 21 – Download Output Files
Download both generated CSV files to your computer.

In [ ]:
files.download(pca_output)
files.download(correlation_output)

## Final Summary

In this notebook, the Week 3 cleaned IPL dataset was used to perform Week 7 dimensionality reduction. Numerical features were selected, missing values were imputed using the median, constant and highly correlated features were removed, and the remaining features were standardized. PCA was then applied to reduce the feature space.

The notebook generates:
- Correlation matrix and heatmap
- Explained variance ratio and cumulative variance plots
- 2D PCA scatter plot, if at least two components are available
- 3D PCA scatter plot, if at least three input features are available
- PCA-reduced dataset CSV
- Correlation matrix CSV

**Important:** Run the notebook from top to bottom in Google Colab. Actual component counts and variance percentages will be calculated from the uploaded dataset; they are not hard-coded.